# StockSense — Phase 1: Exploratory Data Analysis (EDA)
**Project:** Demand Forecasting & Inventory Risk Simulator  
**Dataset:** Walmart Recruiting - Store Sales Forecasting (Kaggle)  
**Unit of Analysis:** Store $\times$ Department Weekly Series  

### Objectives:
1. **Dataset Scope & Cardinality:** Stores, Departments, reporting weeks.
2. **Negative Sales & Returns:** Quantify return rates and audit negative values.
3. **Series Continuity & Filtering:** Filter series with $\ge 104$ weeks (2 full annual cycles).
4. **Holiday Effects & Seasonality:** Measure lift for Super Bowl, Labor Day, Thanksgiving, Christmas.
5. **Top Series by Revenue:** Identify top 20 candidate series for Monte Carlo simulation.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.config import DATA_PROCESSED_DIR, DATA_RAW_DIR
from src.load_data import HOLIDAY_MAP, build_star_schema

print(f"Data Raw: {DATA_RAW_DIR}")
print(f"Data Processed: {DATA_PROCESSED_DIR}")

## 1. Load Data (Parquet Cache or Raw CSVs)

In [ ]:
parquet_path = DATA_PROCESSED_DIR / "fact_sales_weekly.parquet"
if not parquet_path.exists():
    print("Processed parquet not found. Running build_star_schema...")
    dim_store, dim_date, fact_sales, param_inv = build_star_schema(DATA_RAW_DIR)
else:
    print("Loading cached parquet files...")
    fact_sales = pd.read_parquet(DATA_PROCESSED_DIR / "fact_sales_weekly.parquet")
    dim_date = pd.read_parquet(DATA_PROCESSED_DIR / "dim_date.parquet")
    dim_store = pd.read_parquet(DATA_PROCESSED_DIR / "dim_store.parquet")

df = fact_sales.merge(dim_date, on="date_key", how="inner")
df = df.merge(dim_store, on="store_id", how="inner")
df["full_date"] = pd.to_datetime(df["full_date"])
df = df.sort_values(["store_id", "dept_id", "full_date"]).reset_index(drop=True)

print(f"Total Sales Records: {len(df):,}")
print(f"Stores: {df['store_id'].nunique()} | Depts: {df['dept_id'].nunique()} | Weeks: {df['full_date'].nunique()}")
df.head()

## 2. Negative Sales & Returns Analysis

In [ ]:
neg_sales = df[df["weekly_sales"] < 0]
neg_count = len(neg_sales)
total_count = len(df)
neg_pct = (neg_count / total_count) * 100

print(f"Negative Sales Records: {neg_count:,} out of {total_count:,} ({neg_pct:.3f}%)")
if neg_count > 0:
    print(f"Min Sales (Largest Return): ${neg_sales['weekly_sales'].min():,.2f}")
    print(f"Total Return Volume: ${neg_sales['weekly_sales'].sum():,.2f}")
    print(f"Average Return Size: ${neg_sales['weekly_sales'].mean():,.2f}")

## 3. Series Continuity (>= 104 Weeks Filter)

In [ ]:
series_df = df.groupby(["store_id", "dept_id"]).size().reset_index(name="week_count")
total_series = len(series_df)
series_104 = series_df[series_df["week_count"] >= 104]
count_104 = len(series_104)

print(f"Total Series: {total_series:,}")
print(f"Series with >= 104 weeks: {count_104:,} ({(count_104/total_series)*100:.1f}%)")
print(f"Series with full 143 weeks: {(series_df['week_count'] == 143).sum():,}")

## 4. Holiday Lifts

In [ ]:
named = df[df["holiday_name"].notna()].groupby("holiday_name")["weekly_sales"].agg(["count", "mean"]).reset_index()
non_hol_mean = df[df["is_holiday_week"] == 0]["weekly_sales"].mean()
named["lift_pct"] = ((named["mean"] / non_hol_mean) - 1) * 100
named = named.sort_values("mean", ascending=False)
print(named)

## 5. Top 20 Series by Revenue

In [ ]:
df_qual = df.merge(series_104[["store_id", "dept_id"]], on=["store_id", "dept_id"], how="inner")
top20 = df_qual.groupby(["store_id", "dept_id", "store_type"]).agg(
    total_revenue=("weekly_sales", "sum"),
    mean_weekly_sales=("weekly_sales", "mean"),
    std_weekly_sales=("weekly_sales", "std"),
    weeks_available=("weekly_sales", "count")
).reset_index()

top20["cv_demand"] = (top20["std_weekly_sales"] / top20["mean_weekly_sales"]).round(3)
top20 = top20.sort_values("total_revenue", ascending=False).reset_index(drop=True)
top20["revenue_rank"] = top20.index + 1
top20.head(20)